# Modelos clásicos y cuánticos para la predicción de la estabilidad proteica

**Marco Antonio Roca Rodríguez** · Trabajo Fin de Máster · Máster Universitario en Inteligencia Artificial · Universidad Alfonso X el Sabio (UAX)

<a href="https://colab.research.google.com/github/marrocrod/TFM_UAX/blob/main/notebook/estabilidad_proteica.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"></a>

---

## Objetivo

Predecir el cambio de energía libre de plegamiento $\Delta\Delta G$ (kcal/mol) que provoca una **mutación puntual** en una proteína, y comparar tres modelos evaluados sobre las mismas mutaciones y con el mismo protocolo:

| Modelo | Tipo | Implementación |
|---|---|---|
| **XGBoost** | árboles de decisión potenciados | `xgboost` |
| **FCNN** | red neuronal densa | TensorFlow / Keras |
| **QNN** | red híbrida: circuito cuántico variacional de 2 qubits + red clásica | Qiskit + PyTorch |

## Datos y características

- **S2648**: 2 648 mutaciones puntuales de 131 estructuras del PDB con su $\Delta\Delta G$ experimental.
- **XGBoost y FCNN** usan 1 066 variables por mutación:
  - **descriptores** del residuo original y del mutado y de su entorno en la secuencia (±7 posiciones);
  - **diferencias fisicoquímicas** (volumen, hidrofobicidad y flexibilidad);
  - el **embedding de ProtT5** del residuo mutado (1 024 dimensiones).
- **La QNN** usa, por diseño, un vector más ligero de 68 variables, sin ProtT5, para que la entrada al circuito sea pequeña. Tiene codificación *one-hot* de los dos residuos, las mismas diferencias fisicoquímicas y la composición de la ventana ±7.

## Protocolo de evaluación

- **Validación cruzada agrupada por proteína** (5 particiones, `GroupKFold`) y un **conjunto de prueba fijo del 20 %** formado por proteínas que el modelo no ha visto. Los grupos son los *clusters* de proteínas homólogas del propio S2648 (columna `CLID`).
- Todo lo que se aprende de los datos (escalado, PCA, vecinos kNN, parada temprana) se ajusta **solo con la parte de entrenamiento** de cada partición.
- Métricas:
  - **correlación de Pearson (PCC)**, la métrica habitual en la literatura de $\Delta\Delta G$;
  - **RMSE** y **MAE**, en kcal/mol.

## Correcciones respecto a la versión original

| # | Error en la versión original | Corrección |
|---|---|---|
| 1 | Las posiciones de S2648 siguen la numeración del PDB, pero se usaban como índice de la secuencia (`posición − 1`). En cadenas que no empiezan en 1, que tienen huecos o que tienen códigos de inserción, el residuo leído no era el mutado | Se construye el mapa *número de residuo del PDB → índice en la secuencia* y se comprueba que el aminoácido en esa posición coincide con el residuo original de la mutación. Las que no coinciden se descartan y se informa de cuántas son |
| 2 | Las mutaciones con código de inserción (`L27CN`) se leían mal: la `C` se tomaba como residuo mutado | La expresión regular separa posición, código de inserción y residuo mutado |
| 3 | El embedding de ProtT5 se tomaba en `posición + 1`, pero ProtT5 no añade token inicial: se usaba el residuo **siguiente** al mutado | Se toma en el índice correcto. Además, se calcula una vez por cadena (132) en lugar de una por mutación (2 648) |
| 4 | El recuento de aminoácidos vecinos usaba el índice alfabético de la letra (`ord(aa) − 65`), así que V, W e Y nunca se contaban y sí letras que no son aminoácidos | Recuento sobre los 20 aminoácidos estándar |
| 5 | Las particiones eran por mutación: mutaciones de la misma proteína caían en entrenamiento y en prueba, lo que infla las métricas | Particiones agrupadas por *cluster* de proteína (`GroupKFold` y `GroupShuffleSplit`) |
| 6 | El escalado (y en la QNN también la PCA) se ajustaba con todos los datos antes de dividir | Se ajusta dentro de cada partición, solo con entrenamiento |
| 7 | La QNN usaba el conjunto de prueba para la parada temprana y para el planificador de la tasa de aprendizaje | Se separa un conjunto de validación del entrenamiento, agrupado por proteína |
| 8 | La QNN repetía por su cuenta la descarga y el filtrado, así que trabajaba con otro subconjunto de mutaciones (2 464 frente a 2 647) y con otras particiones. Además, su variable de posición era el número de residuo del PDB sin mapear | Los tres modelos usan las mismas mutaciones y las mismas particiones. La QNN conserva su vector propio sin ProtT5, como en la memoria, calculado sobre la posición corregida. `QNN_USE_PROTT5` permite añadirle ProtT5 |
| 9 | XGBoost se evaluaba en validación cruzada con una configuración (300 árboles, profundidad 4) y en la prueba fija con otra (400, profundidad 5) | Una única configuración para las dos evaluaciones |
| 10 | La validación interna de la FCNN y los vecinos kNN se separaban por mutación | Se separan por proteína |
| 11 | Rutas fijas de Colab (`/content`, `/mnt/data`), dos descargas del PDB e instalaciones repartidas por el notebook | Configuración centralizada, una sola descarga con caché y una celda de instalación |
| 12 | Algunas estructuras ya no existen en formato PDB (p. ej. 2A01) y fallaban | Si el formato PDB no está disponible, se usa mmCIF |

Los modelos, las pérdidas y los hiperparámetros **no cambian**. La mejora de métricas y de rendimiento es una fase posterior.

## Mapa del notebook

| Parte | Contenido |
|---|---|
| I | Entorno y configuración |
| II | Datos: S2648, estructuras y secuencias |
| III | Características |
| IV | Protocolo de evaluación |
| V | XGBoost |
| VI | FCNN |
| VII | QNN |
| VIII | Comparación y resultados |

---
# Parte I — Entorno y configuración

### 1 · Dependencias

En Google Colab basta con ejecutar esta celda: si el notebook se ha abierto desde GitHub, clona el repositorio para tener los datos, e instala lo que falte (TensorFlow, PyTorch y scikit-learn ya vienen instalados). En local, usa `requirements.txt`. ProtT5-XL necesita GPU para que la extracción de embeddings no tarde horas: en Colab, *Entorno de ejecución → Cambiar tipo de entorno → GPU*.

In [ ]:
import importlib.util, os, subprocess, sys

# On Colab the notebook is opened from GitHub without the rest of the repository:
# clone it once and work from its notebook/ folder so that relative paths resolve.
if "google.colab" in sys.modules and not os.path.exists("../data/s2648.csv"):
    if not os.path.exists("/content/TFM_UAX"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/marrocrod/TFM_UAX", "/content/TFM_UAX"],
                       check=True)
    os.chdir("/content/TFM_UAX/notebook")

REQUIRED = {"Bio": "biopython", "xgboost": "xgboost", "transformers": "transformers",
            "sentencepiece": "sentencepiece", "qiskit": "qiskit>=1.2",
            "qiskit_machine_learning": "qiskit-machine-learning>=0.8"}
missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Dependencias listas" + (f" (instaladas: {', '.join(missing)})" if missing else ""))

### 2 · Configuración

Punto único de ajuste. `RUN_MODE = "smoke"` reduce los datos y las épocas para comprobar en pocos minutos que todo el pipeline funciona; `"full"` es la ejecución completa. Los hiperparámetros de los tres modelos son los de la versión original.

In [ ]:
import os, json, math, random, re, time, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# PyTorch must load its compiler stack (and the LLVM bundled with Triton) before
# TensorFlow loads its own: otherwise the first torch optimizer created after the
# FCNN aborts the kernel with "LLVM ERROR: inconsistency in registered CommandLine options".
import torch
import torch._dynamo  # noqa: F401

# ── Run mode ────────────────────────────────────────────────────────────────
RUN_MODE = "full"                  # "smoke" | "full"
SEED     = 42

# ── Paths (relative to notebook/; on Colab the repository is cloned first) ──
DATA_CSV    = "../data/s2648.csv"
CACHE_DIR   = "../cache"           # PDB files and ProtT5 embeddings (not versioned)
RESULTS_DIR = "../results"
PDB_DIR     = os.path.join(CACHE_DIR, "pdb")
EMB_DIR     = os.path.join(CACHE_DIR, "prott5")
for d in (PDB_DIR, EMB_DIR, RESULTS_DIR):
    os.makedirs(d, exist_ok=True)

# ── Features ────────────────────────────────────────────────────────────────
PROTT5_MODEL = "Rostlab/prot_t5_xl_bfd"
WINDOW       = 7                   # ± sequence neighbours around the mutation

# ── Evaluation protocol ─────────────────────────────────────────────────────
N_FOLDS   = 5                      # grouped cross-validation
TEST_SIZE = 0.20                   # fixed held-out set of unseen protein clusters
GROUP_COL = "CLID"                 # homology cluster of each protein in S2648

# ── XGBoost (single configuration for CV and held-out test) ─────────────────
XGB_PARAMS = dict(n_estimators=400, max_depth=5, learning_rate=0.05,
                  subsample=1.0, colsample_bytree=1.0)

# ── FCNN ────────────────────────────────────────────────────────────────────
FCNN_SEEDS         = [42, 7, 1337]  # ensemble of three seeds
FCNN_VAL_SIZE      = 0.10           # share of the training proteins used for early stopping
FCNN_BATCH_SIZE    = 128
FCNN_EPOCHS        = 250
FCNN_PATIENCE      = 16
FCNN_L2            = 1e-5
FCNN_DROPOUT       = 0.40
FCNN_KNN_NEIGHBORS = 28
FCNN_LOSS_WEIGHTS  = (0.50, 0.45, 0.05)   # MSE, (1 - r²), std matching

# ── QNN ─────────────────────────────────────────────────────────────────────
QNN_TARGET_DIM    = 16             # PCA components fed to the circuit
QNN_QUBITS        = 2
QNN_BATCH_SIZE    = 64
QNN_EPOCHS_CV     = 12
QNN_EPOCHS_TEST   = 18
QNN_PATIENCE      = 5
QNN_LR            = 2e-3
QNN_WD            = 1e-3
QNN_VAL_SIZE      = 0.10
QNN_KNN_NEIGHBORS = 15
QNN_LOSS_WEIGHTS  = (0.55, 0.40, 0.05)    # Huber, (1 - r²), std matching
QNN_USE_PROTT5    = False          # design of the thesis: lighter input without ProtT5

if RUN_MODE == "smoke":
    FCNN_SEEDS, FCNN_EPOCHS = [42], 3
    QNN_EPOCHS_CV, QNN_EPOCHS_TEST, QNN_TARGET_DIM = 1, 1, 4


def set_global_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)


set_global_seed(SEED)
print(f"Modo: {RUN_MODE.upper()} | semilla {SEED} | validación {N_FOLDS}-fold agrupada por '{GROUP_COL}'")

---
# Parte II — Datos

### 3 · Carga de S2648

Cada fila es una mutación: estructura (`PDB`, `CHAIN`), mutación (`MUT`, p. ej. `C30S`), $\Delta\Delta G$ experimental (`DDG`, kcal/mol), condiciones del experimento (`pH`, `T`) y el *cluster* de homología de la proteína (`CLID`).

La posición sigue la **numeración del PDB**, que puede llevar **código de inserción**, como en `L27CN`: residuo L en la posición 27C, mutado a N. La expresión regular separa los cuatro campos.

In [ ]:
df = pd.read_csv(DATA_CSV)
parts = df["MUT"].str.extract(r"^([A-Z])(\d+)([A-Z]?)([A-Z])$")
parts.columns = ["wt_res", "position", "icode", "mut_res"]
df = pd.concat([df, parts], axis=1)
df["position"] = df["position"].astype(int)
df["icode"] = df["icode"].fillna("").replace("", " ")     # Biopython uses " " for "no insertion code"

if RUN_MODE == "smoke":
    keep = df[GROUP_COL].drop_duplicates().sample(12, random_state=SEED)
    df = df[df[GROUP_COL].isin(keep)].reset_index(drop=True)

print(f"Mutaciones: {len(df):,} | estructuras: {df['PDB'].nunique()} | clusters: {df[GROUP_COL].nunique()}")
print(f"Con código de inserción: {(df['icode'] != ' ').sum()}")
print(f"ΔΔG: media {df['DDG'].mean():.2f} ± {df['DDG'].std():.2f} kcal/mol "
      f"(rango {df['DDG'].min():.1f} a {df['DDG'].max():.1f})")

### 4 · Estructuras y secuencias

Para cada cadena se descarga la estructura (formato PDB y, si no existe, mmCIF) y se recorren sus residuos estándar en orden. Así se obtienen dos cosas:

- la **secuencia** de la cadena, que es la entrada de ProtT5;
- el **mapa** *(número de residuo del PDB, código de inserción) → índice en la secuencia*.

Con ese mapa se localiza cada mutación y se **comprueba que el aminoácido en esa posición es el residuo original** que indica S2648. Las mutaciones que no se pueden localizar o cuyo residuo no coincide se descartan, y la celda informa de cuántas hay.

In [ ]:
from Bio.PDB import PDBList, PDBParser, MMCIFParser
from Bio.PDB.Polypeptide import is_aa
from Bio.SeqUtils import seq1

warnings.filterwarnings("ignore", module="Bio")
_pdbl = PDBList(verbose=False)


def fetch_structure(pdb_id):
    """Download (or reuse) a structure, trying PDB format first and mmCIF second."""
    pdb_id = pdb_id.lower()
    for fmt, parser in (("pdb", PDBParser(QUIET=True)), ("mmCif", MMCIFParser(QUIET=True))):
        try:
            path = _pdbl.retrieve_pdb_file(pdb_id, pdir=PDB_DIR, file_format=fmt)
            if os.path.exists(path):
                return parser.get_structure(pdb_id, path)
        except Exception:
            continue
    return None


def chain_sequence_and_map(structure, chain_id):
    """Sequence of the standard residues of a chain (first model) and the map
    (residue number, insertion code) -> index in that sequence."""
    model = next(iter(structure))
    if chain_id not in model:
        return None, None
    seq, index = [], {}
    for res in model[chain_id]:
        if res.id[0] != " " or not is_aa(res, standard=True):     # skip waters, ligands, non-standard
            continue
        index[(res.id[1], res.id[2])] = len(seq)
        seq.append(seq1(res.get_resname()))
    return "".join(seq), index


chains = {}                                                       # (pdb, chain) -> (sequence, index map)
for pdb_id, chain_id in df[["PDB", "CHAIN"]].drop_duplicates().itertuples(index=False):
    st = fetch_structure(pdb_id)
    if st is None:
        print(f"  ⚠ {pdb_id}: estructura no disponible")
        continue
    seq, idx_map = chain_sequence_and_map(st, chain_id)
    if seq:
        chains[(pdb_id, chain_id)] = (seq, idx_map)
    else:
        print(f"  ⚠ {pdb_id}: cadena {chain_id} no encontrada")

# ── Locate every mutation and check the wild-type residue ───────────────────
seq_index, status = [], []
for r in df.itertuples(index=False):
    entry = chains.get((r.PDB, r.CHAIN))
    if entry is None:
        seq_index.append(-1); status.append("sin estructura"); continue
    seq, idx_map = entry
    k = idx_map.get((r.position, r.icode))
    if k is None:
        seq_index.append(-1); status.append("posición no encontrada"); continue
    seq_index.append(k)
    status.append("ok" if seq[k] == r.wt_res else "residuo distinto")
df["seq_index"], df["status"] = seq_index, status

print(f"Cadenas cargadas: {len(chains)} de {df[['PDB', 'CHAIN']].drop_duplicates().shape[0]}")
print(df["status"].value_counts().to_string())

# How many mutations the original indexing (position - 1) would have read correctly
ok_old = sum(1 for r in df.itertuples(index=False) if (r.PDB, r.CHAIN) in chains
             and 0 <= r.position - 1 < len(chains[(r.PDB, r.CHAIN)][0])
             and chains[(r.PDB, r.CHAIN)][0][r.position - 1] == r.wt_res)
print(f"\nCon el índice original (posición − 1) el residuo coincidía en {ok_old} de {len(df)} mutaciones "
      f"({100 * ok_old / len(df):.1f} %).")

df = df[df["status"] == "ok"].reset_index(drop=True)
print(f"Mutaciones válidas: {len(df):,}")

---
# Parte III — Características

### 5 · Descriptores de la mutación

Se mantienen los descriptores de la versión original, con el recuento de vecinos corregido:

| Bloque | Dim. | Contenido |
|---|---:|---|
| Residuos | 2 | código del residuo original y del mutado (`ord(letra) − 65`) |
| Posición | 2 | índice en la secuencia y posición relativa (índice / longitud) |
| Vecinos | 14 | código de cada vecino a ±7 posiciones (−1 si cae fuera de la cadena) |
| Composición | 20 | recuento de cada aminoácido estándar en esa ventana |
| Fisicoquímica | 3 | Δ volumen (Å³), Δ hidrofobicidad y Δ flexibilidad (mutado − original) |
| Tipo | 1 | identificador del par original → mutado (1 – 380) |

In [ ]:
AA20 = "ACDEFGHIKLMNPQRSTVWY"
AA_TO_IDX = {a: i for i, a in enumerate(AA20)}

AA_VOLUME = {'A': 88.6, 'R': 173.4, 'N': 114.1, 'D': 111.1, 'C': 108.5, 'E': 138.4, 'Q': 143.8, 'G': 60.1,
             'H': 153.2, 'I': 166.7, 'L': 166.7, 'K': 168.6, 'M': 162.9, 'F': 189.9, 'P': 112.7, 'S': 89.0,
             'T': 116.1, 'W': 227.8, 'Y': 193.6, 'V': 140.0}
AA_HYDROPHOBICITY = {'A': 0, 'R': 3.71, 'N': 3.47, 'D': 2.95, 'C': 0.49, 'E': 1.64, 'Q': 3.01, 'G': 1.72,
                     'H': 4.76, 'I': -1.56, 'L': -1.81, 'K': 5.39, 'M': -0.76, 'F': -2.2, 'P': -1.52,
                     'S': 1.83, 'T': 1.78, 'W': -0.38, 'Y': -1.09, 'V': -0.78}
AA_FLEXIBILITY = {'A': 1, 'R': 81, 'N': 36, 'D': 18, 'C': 3, 'E': 54, 'Q': 108, 'G': 1, 'H': 36, 'I': 9,
                  'L': 9, 'K': 81, 'M': 27, 'F': 18, 'P': 2, 'S': 3, 'T': 3, 'W': 36, 'Y': 18, 'V': 3}
MUTATION_ORDER = ['A', 'F', 'C', 'D', 'N', 'E', 'Q', 'G', 'H', 'L', 'I', 'K', 'M', 'P', 'R', 'S', 'T', 'V', 'W', 'Y']


def mutation_type(wt, mt):
    """Identifier 1..380 of the ordered pair (wt, mt); same numbering as the original notebook."""
    if wt not in MUTATION_ORDER or mt not in MUTATION_ORDER or wt == mt:
        return 0
    wi, mi = MUTATION_ORDER.index(wt), MUTATION_ORDER.index(mt)
    return wi * 19 + (mi - (1 if mi > wi else 0)) + 1


def handcrafted_features(seq, k, wt, mt, window=WINDOW):
    vec = [ord(wt) - 65, ord(mt) - 65, k, k / len(seq)]
    composition = [0] * 20
    for off in range(-window, window + 1):
        if off == 0:
            continue
        i = k + off
        if 0 <= i < len(seq):
            vec.append(ord(seq[i]) - 65)
            if seq[i] in AA_TO_IDX:                      # fixed: count by amino acid, not by letter index
                composition[AA_TO_IDX[seq[i]]] += 1
        else:
            vec.append(-1)
    vec += composition
    vec += [AA_VOLUME[mt] - AA_VOLUME[wt],
            AA_HYDROPHOBICITY[mt] - AA_HYDROPHOBICITY[wt],
            AA_FLEXIBILITY[mt] - AA_FLEXIBILITY[wt],
            mutation_type(wt, mt)]
    return vec


HANDCRAFTED_NAMES = (["wt_code", "mut_code", "seq_index", "rel_position"]
                     + [f"neighbour_{o:+d}" for o in range(-WINDOW, WINDOW + 1) if o != 0]
                     + [f"count_{a}" for a in AA20]
                     + ["d_volume", "d_hydrophobicity", "d_flexibility", "mutation_type"])
X_hand = np.array([handcrafted_features(chains[(r.PDB, r.CHAIN)][0], r.seq_index, r.wt_res, r.mut_res)
                   for r in df.itertuples(index=False)], dtype=np.float32)
print(f"Descriptores: {X_hand.shape}")

### 6 · Embeddings de ProtT5

**ProtT5-XL (BFD)** es un modelo de lenguaje de proteínas: para cada residuo devuelve un vector de 1 024 números que resume su contexto en la secuencia. Se usa el vector **del residuo mutado**.

ProtT5 recibe los aminoácidos separados por espacios y **solo añade un token final** (`</s>`), sin token inicial. Por eso el residuo $k$ de la secuencia corresponde a la salida $k$, no a la $k+1$ que usaba la versión original.

Cada cadena se procesa **una sola vez** y su matriz $L \times 1024$ se guarda en `cache/prott5/`.

In [ ]:
import torch
from transformers import T5Tokenizer, T5EncoderModel

device = "cuda" if torch.cuda.is_available() else "cpu"
_t5 = {}


def prott5_chain_embedding(pdb_id, chain_id, seq):
    """(L, 1024) per-residue ProtT5 embedding of a chain, cached on disk."""
    path = os.path.join(EMB_DIR, f"{pdb_id}_{chain_id}.npy")
    if os.path.exists(path):
        return np.load(path)
    if not _t5:
        _t5["tok"] = T5Tokenizer.from_pretrained(PROTT5_MODEL, do_lower_case=False)
        model = T5EncoderModel.from_pretrained(PROTT5_MODEL).to(device).eval()
        _t5["model"] = model.half() if device == "cuda" else model
    seq_in = " ".join(re.sub(r"[UZOB]", "X", seq))           # ProtT5 convention for rare residues
    ids = _t5["tok"](seq_in, return_tensors="pt", add_special_tokens=True).to(device)
    with torch.no_grad():
        out = _t5["model"](input_ids=ids["input_ids"], attention_mask=ids["attention_mask"]).last_hidden_state
    assert out.shape[1] == len(seq) + 1, "expected one token per residue plus the final </s>"
    emb = out[0, :len(seq)].float().cpu().numpy()            # drop the trailing </s>; residue k -> row k
    np.save(path, emb)
    return emb


t0 = time.time()
emb_by_chain = {key: prott5_chain_embedding(key[0], key[1], seq)
                for key, (seq, _) in chains.items() if key in set(zip(df["PDB"], df["CHAIN"]))}
X_emb = np.vstack([emb_by_chain[(r.PDB, r.CHAIN)][r.seq_index] for r in df.itertuples(index=False)]).astype(np.float32)
print(f"Embeddings ProtT5: {X_emb.shape} ({len(emb_by_chain)} cadenas, {time.time() - t0:.0f} s, dispositivo {device})")

if "model" in _t5:                                           # free GPU memory for the models below
    del _t5["model"]
    torch.cuda.empty_cache() if device == "cuda" else None

### 7 · Matrices finales

- **`X` (1 066):** descriptores + ProtT5, para XGBoost y la FCNN.
- **`X_qnn` (68):** el vector propio de la QNN, el mismo que en la memoria. Lleva propiedades básicas de la mutación, codificación *one-hot* del residuo original y del mutado (20 + 20), posición normalizada y composición de la ventana ±7. No usa ProtT5, para mantener pequeña la entrada al circuito. Con `QNN_USE_PROTT5 = True` se le añade el embedding.

Las dos matrices tienen **las mismas filas**, en el mismo orden, así que los tres modelos comparten mutaciones, objetivo `y`, grupos y particiones.

In [ ]:
X = np.hstack([X_hand, X_emb]).astype(np.float32)
y = df["DDG"].to_numpy(np.float32)
groups = df[GROUP_COL].to_numpy()
FEATURE_NAMES = HANDCRAFTED_NAMES + [f"prott5_{i}" for i in range(X_emb.shape[1])]


def qnn_features(seq, k, wt, mt, window=WINDOW):
    """68-d input of the QNN (thesis design): basic properties, one-hot of both residues,
    normalised position and amino-acid composition of the ±window neighbourhood."""
    basic = [ord(wt) - 65, ord(mt) - 65, k,
             AA_VOLUME[mt] - AA_VOLUME[wt], AA_HYDROPHOBICITY[mt] - AA_HYDROPHOBICITY[wt],
             AA_FLEXIBILITY[mt] - AA_FLEXIBILITY[wt], mutation_type(wt, mt)]
    onehot_wt, onehot_mt = [0] * 20, [0] * 20
    onehot_wt[MUTATION_ORDER.index(wt)] = 1
    onehot_mt[MUTATION_ORDER.index(mt)] = 1
    counts = [0] * 20
    for off in range(-window, window + 1):
        i = k + off
        if off != 0 and 0 <= i < len(seq) and seq[i] in MUTATION_ORDER:
            counts[MUTATION_ORDER.index(seq[i])] += 1
    return basic + onehot_wt + onehot_mt + [k / max(1, len(seq) - 1)] + counts


X_qnn = np.array([qnn_features(chains[(r.PDB, r.CHAIN)][0], r.seq_index, r.wt_res, r.mut_res)
                  for r in df.itertuples(index=False)], dtype=np.float32)
if QNN_USE_PROTT5:
    X_qnn = np.hstack([X_qnn, X_emb]).astype(np.float32)

print(f"X    : {X.shape} = {X_hand.shape[1]} descriptores + {X_emb.shape[1]} ProtT5   (XGBoost, FCNN)")
print(f"X_qnn: {X_qnn.shape}{'  (+ ProtT5)' if QNN_USE_PROTT5 else ''}   (QNN)")
print(f"y: {y.shape} | grupos: {len(np.unique(groups))}")

---
# Parte IV — Protocolo de evaluación

### 8 · Particiones, métricas y registro

- **Validación cruzada**: `GroupKFold(5)` sobre los *clusters* de proteína. Ninguna proteína aparece a la vez en entrenamiento y en evaluación.
- **Prueba fija**: `GroupShuffleSplit` con un 20 % de los *clusters*, fijado con la semilla.
- Cada modelo se define como una función `fit_predict(X_tr, y_tr, g_tr, X_te)` que hace **todo** su preprocesado dentro. Así es imposible que la parte de evaluación influya en el entrenamiento.
- `evaluate()` ejecuta la validación cruzada y la prueba fija, imprime las métricas y guarda las predicciones para la Parte VIII.
- `knn_oof_feature()` calcula la meta-característica kNN que usan la FCNN y la QNN: predicción *out-of-fold* (por grupos) en el entrenamiento y ajuste completo para la evaluación.

In [ ]:
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.neighbors import KNeighborsRegressor
from scipy.stats import pearsonr

CV_SPLITS = list(GroupKFold(n_splits=N_FOLDS).split(X, y, groups))
TRAIN_IDX, TEST_IDX = next(GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEED)
                           .split(X, y, groups))
assert not set(groups[TRAIN_IDX]) & set(groups[TEST_IDX]), "a protein cluster is on both sides"
print(f"Prueba fija: {len(TEST_IDX)} mutaciones de {len(set(groups[TEST_IDX]))} clusters "
      f"| entrenamiento: {len(TRAIN_IDX)} de {len(set(groups[TRAIN_IDX]))}")


def grouped_holdout(n, g, size, seed=SEED):
    """Indices (train, val) of a split of n samples that keeps every protein cluster on one side."""
    return next(GroupShuffleSplit(n_splits=1, test_size=size, random_state=seed).split(np.arange(n), groups=g))


def regression_metrics(y_true, y_pred):
    pcc = pearsonr(y_true, y_pred)[0] if np.std(y_pred) > 0 else 0.0
    mse = mean_squared_error(y_true, y_pred)
    return {"PCC": float(pcc), "RMSE": float(np.sqrt(mse)), "MSE": float(mse),
            "MAE": float(mean_absolute_error(y_true, y_pred))}


def knn_oof_feature(X_tr, y_tr, g_tr, X_te, n_neighbors):
    """kNN prediction as a meta-feature: out-of-fold (grouped) on train, full fit for test."""
    oof = np.zeros(len(X_tr), dtype=np.float32)
    n_splits = min(5, len(np.unique(g_tr)))
    for tr, va in GroupKFold(n_splits=n_splits).split(X_tr, y_tr, g_tr):
        knn = KNeighborsRegressor(n_neighbors=min(n_neighbors, len(tr)), weights="distance")
        oof[va] = knn.fit(X_tr[tr], y_tr[tr]).predict(X_tr[va])
    knn = KNeighborsRegressor(n_neighbors=min(n_neighbors, len(X_tr)), weights="distance").fit(X_tr, y_tr)
    return oof.reshape(-1, 1), knn.predict(X_te).astype(np.float32).reshape(-1, 1)


RESULTS = {}


def evaluate(name, fit_predict, features=None, test_kwargs=None):
    """Grouped 5-fold CV + fixed held-out set for one model.
    features: feature matrix with the same rows as X (default X).
    test_kwargs are passed only to the held-out fit (e.g. more epochs)."""
    F = X if features is None else features
    folds, oof = [], np.zeros_like(y)
    for f, (tr, te) in enumerate(CV_SPLITS, 1):
        t0 = time.time()
        pred = fit_predict(F[tr], y[tr], groups[tr], F[te])
        oof[te] = pred
        m = regression_metrics(y[te], pred)
        folds.append(m)
        print(f"  fold {f}: PCC={m['PCC']:.3f}  RMSE={m['RMSE']:.3f}  MAE={m['MAE']:.3f}  ({time.time() - t0:.0f} s)")
    cv = {k: (float(np.mean([m[k] for m in folds])), float(np.std([m[k] for m in folds]))) for k in folds[0]}
    print(f"  CV  : PCC={cv['PCC'][0]:.3f} ± {cv['PCC'][1]:.3f} | RMSE={cv['RMSE'][0]:.3f} ± {cv['RMSE'][1]:.3f} "
          f"| MAE={cv['MAE'][0]:.3f} ± {cv['MAE'][1]:.3f}")

    pred_test = fit_predict(F[TRAIN_IDX], y[TRAIN_IDX], groups[TRAIN_IDX], F[TEST_IDX], **(test_kwargs or {}))
    test = regression_metrics(y[TEST_IDX], pred_test)
    print(f"  TEST: PCC={test['PCC']:.3f} | RMSE={test['RMSE']:.3f} | MAE={test['MAE']:.3f}")

    RESULTS[name] = {"cv_folds": folds, "cv": cv, "test": test,
                     "pred_test": pred_test.tolist(), "pred_oof": oof.tolist()}
    return RESULTS[name]

---
# Parte V — XGBoost

### 9 · XGBoost

Árboles de decisión potenciados por gradiente, con la configuración final de la versión original: 400 árboles, profundidad 5 y tasa 0,05. Los árboles no dependen de la escala de las variables, así que no se estandariza. Al final se dibujan las curvas de error de la prueba fija.

In [ ]:
from xgboost import XGBRegressor


def xgb_fit_predict(X_tr, y_tr, g_tr, X_te, return_model=False):
    model = XGBRegressor(**XGB_PARAMS, random_state=SEED, eval_metric=["rmse", "mae"])
    model.fit(X_tr, y_tr, verbose=False)
    pred = model.predict(X_te)
    return (pred, model) if return_model else pred


print("XGBoost")
evaluate("XGBoost", xgb_fit_predict)

# Learning curves on the fixed split (diagnostic only: no early stopping)
model = XGBRegressor(**XGB_PARAMS, random_state=SEED, eval_metric=["rmse", "mae"])
model.fit(X[TRAIN_IDX], y[TRAIN_IDX],
          eval_set=[(X[TRAIN_IDX], y[TRAIN_IDX]), (X[TEST_IDX], y[TEST_IDX])], verbose=False)
res = model.evals_result()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric in zip(axes, ["rmse", "mae"]):
    ax.plot(res["validation_0"][metric], label="entrenamiento")
    ax.plot(res["validation_1"][metric], label="prueba")
    ax.set_xlabel("Árbol"); ax.set_ylabel(metric.upper()); ax.grid(alpha=0.3); ax.legend()
    ax.set_title(f"XGBoost — {metric.upper()} por número de árboles")
plt.tight_layout(); plt.show()

---
# Parte VI — FCNN

### 10 · Red neuronal densa

Arquitectura y entrenamiento de la versión original:

- **Red:** `LayerNorm → Dense 320 → Dense 384 → Dense 256 → 1`, con activación *swish*, *dropout* 0,4 y regularización L2.
- **Meta-característica kNN:** la predicción de un regresor de 28 vecinos, calculada *out-of-fold* dentro del entrenamiento.
- **Pérdida compuesta** sobre $y$ estandarizada: $0{,}50\,\mathrm{MSE} + 0{,}45\,(1-r^2) + 0{,}05\,(\sigma_{\hat y}-1)^2$.
- **Optimizador:** Adam con *Cosine Decay Restarts*.
- **Ensemble** de 3 semillas.

Lo que cambia es dónde se ajusta cada cosa. El escalado, el kNN y la parada temprana usan solo el entrenamiento de cada partición, y tanto la validación interna como los pliegues del kNN se separan por proteína.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, callbacks, initializers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers.schedules import CosineDecayRestarts
tf.get_logger().setLevel("ERROR")


def pcc_tf(y_true, y_pred, eps=1e-8):
    yt = y_true - tf.reduce_mean(y_true)
    yp = y_pred - tf.reduce_mean(y_pred)
    cov = tf.reduce_mean(yt * yp)
    return tf.clip_by_value(cov / (tf.sqrt(tf.reduce_mean(yt ** 2) + eps) * tf.sqrt(tf.reduce_mean(yp ** 2) + eps) + eps),
                            -1.0, 1.0)


def composite_loss_tf(w_mse, w_corr, w_std):
    def loss(y_true, y_pred):
        mse = tf.reduce_mean(tf.square(y_true - y_pred))
        corr = 1.0 - tf.square(pcc_tf(y_true, y_pred))
        std = tf.square(tf.math.reduce_std(y_pred) - 1.0)
        return w_mse * mse + w_corr * corr + w_std * std
    return loss


def build_fcnn(input_dim, seed):
    init = initializers.GlorotUniform(seed=seed)
    x_in = layers.Input(shape=(input_dim,))
    x = layers.LayerNormalization()(x_in)
    for units, drop in ((320, True), (384, True), (256, False)):
        x = layers.Dense(units, kernel_initializer=init, kernel_regularizer=regularizers.l2(FCNN_L2))(x)
        x = layers.Activation("swish")(x)
        if drop:
            x = layers.Dropout(FCNN_DROPOUT, seed=seed)(x)
    model = models.Model(x_in, layers.Dense(1, kernel_initializer=init)(x))
    schedule = CosineDecayRestarts(initial_learning_rate=5e-4, first_decay_steps=400,
                                   t_mul=1.5, m_mul=0.9, alpha=1e-5)
    model.compile(optimizer=Adam(learning_rate=schedule, clipnorm=1.0),
                  loss=composite_loss_tf(*FCNN_LOSS_WEIGHTS), metrics=["mae", pcc_tf])
    return model


FCNN_HISTORIES = []


def fcnn_fit_predict(X_tr, y_tr, g_tr, X_te):
    # 1) Scaling fitted on the training part only
    sx = StandardScaler().fit(X_tr)
    X_tr, X_te = sx.transform(X_tr), sx.transform(X_te)
    # 2) kNN meta-feature (grouped out-of-fold)
    k_tr, k_te = knn_oof_feature(X_tr, y_tr, g_tr, X_te, FCNN_KNN_NEIGHBORS)
    sk = StandardScaler().fit(k_tr)
    X_tr = np.hstack([X_tr, sk.transform(k_tr)]).astype(np.float32)
    X_te = np.hstack([X_te, sk.transform(k_te)]).astype(np.float32)
    # 3) Standardised target
    sy = StandardScaler().fit(y_tr.reshape(-1, 1))
    y_tr_s = sy.transform(y_tr.reshape(-1, 1)).ravel().astype(np.float32)
    # 4) Validation split by protein cluster, for early stopping
    tr, va = grouped_holdout(len(X_tr), g_tr, FCNN_VAL_SIZE)

    preds, FCNN_HISTORIES[:] = [], []
    for sd in FCNN_SEEDS:
        set_global_seed(sd); tf.random.set_seed(sd)
        model = build_fcnn(X_tr.shape[1], sd)
        h = model.fit(X_tr[tr], y_tr_s[tr], validation_data=(X_tr[va], y_tr_s[va]),
                      epochs=FCNN_EPOCHS, batch_size=FCNN_BATCH_SIZE, shuffle=True, verbose=0,
                      callbacks=[callbacks.EarlyStopping(monitor="val_loss", patience=FCNN_PATIENCE,
                                                         restore_best_weights=True),
                                 callbacks.TerminateOnNaN()])
        FCNN_HISTORIES.append(h.history)
        preds.append(sy.inverse_transform(model.predict(X_te, verbose=0).reshape(-1, 1)).ravel())
    return np.mean(preds, axis=0)


print("FCNN")
evaluate("FCNN", fcnn_fit_predict)

# Loss curves of the last fit (fixed split), one line per seed
plt.figure(figsize=(7, 4.5))
for sd, h in zip(FCNN_SEEDS, FCNN_HISTORIES):
    plt.plot(h["loss"], label=f"entrenamiento (semilla {sd})")
    plt.plot(h["val_loss"], "--", label=f"validación (semilla {sd})")
plt.xlabel("Época"); plt.ylabel("Pérdida compuesta"); plt.title("FCNN — curvas de pérdida (prueba fija)")
plt.grid(alpha=0.3); plt.legend(fontsize=8, ncol=2); plt.tight_layout(); plt.show()

---
# Parte VII — QNN

### 11 · Red neuronal cuántica híbrida

**Entrada del circuito.** El vector `X_qnn` de cada mutación (68 variables, sin ProtT5) se estandariza y se reduce por PCA a 16 componentes. Las dos transformaciones se ajustan solo con el entrenamiento de cada partición.

**Circuito (2 qubits, 8 bloques).** Cada bloque $b$:
1. Codifica dos componentes como ángulos, $R_Y(x_{2b})$ y $R_Y(x_{2b+1})$. Es *data re-uploading*: las 16 componentes entran por pares a lo largo de los 8 bloques.
2. Entrelaza los dos qubits con una CNOT.
3. Aplica rotaciones entrenables $R_Z(w^z)$ y $R_Y(w^y)$ en cada qubit.

Se miden $\langle Z_0\rangle$, $\langle Z_1\rangle$ y $\langle Z_0Z_1\rangle$ de forma exacta con `StatevectorEstimator`, y el circuito tiene **32 parámetros**.

**Red híbrida.** Las 3 medidas se concatenan con la rama clásica, formada por las 16 componentes PCA y la meta-característica kNN, y pasan por un MLP `→ 64 → 64 → 1`. La pérdida compuesta es $0{,}55\,\mathrm{Huber} + 0{,}40\,(1-r^2) + 0{,}05\,(\sigma_{\hat y}-1)^2$. El optimizador es AdamW con `ReduceLROnPlateau`.

**Corrección clave.** La parada temprana y el planificador de la tasa de aprendizaje usan un **conjunto de validación separado del entrenamiento**, agrupado por proteína, y no el conjunto de prueba.

> La simulación con `EstimatorQNN` es lenta: en la versión original, cada época tardaba unos 20 minutos en Colab. Acelerarla es parte de la fase de mejora del rendimiento.

In [ ]:
import torch.nn as nn
from sklearn.decomposition import PCA
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector

warnings.filterwarnings("ignore", module="qiskit_machine_learning")


def build_circuit(n_qubits, n_features):
    """Re-uploading circuit: per block RY(x) on each qubit, a CNOT chain, then trainable RZ·RY."""
    blocks = math.ceil(n_features / n_qubits)
    x  = ParameterVector("x",  blocks * n_qubits)
    wz = ParameterVector("wz", blocks * n_qubits)
    wy = ParameterVector("wy", blocks * n_qubits)
    qc = QuantumCircuit(n_qubits)
    for b in range(blocks):
        for q in range(n_qubits):
            qc.ry(x[b * n_qubits + q], q)
        for q in range(n_qubits - 1):
            qc.cx(q, q + 1)
        for q in range(n_qubits):
            qc.rz(wz[b * n_qubits + q], q)
            qc.ry(wy[b * n_qubits + q], q)
    labels = ["I" * i + "Z" + "I" * (n_qubits - i - 1) for i in range(n_qubits)]
    labels += ["".join("Z" if k in (i, j) else "I" for k in range(n_qubits))
               for i in range(n_qubits) for j in range(i + 1, n_qubits)]
    observables = [SparsePauliOp(lbl) for lbl in labels]
    return qc, list(x), list(wz) + list(wy), observables


def build_qnn(n_qubits, n_features):
    qc, x_params, w_params, observables = build_circuit(n_qubits, n_features)
    qnn = EstimatorQNN(circuit=qc, observables=observables, input_params=x_params,
                       weight_params=w_params, estimator=StatevectorEstimator(), input_gradients=False)
    return qnn, len(observables)


class HybridRegressor(nn.Module):
    """Quantum branch (3 expectation values) concatenated with the classical branch."""
    def __init__(self, qnn, q_out, classical_dim):
        super().__init__()
        self.q = TorchConnector(qnn)
        self.mlp = nn.Sequential(nn.Linear(q_out + classical_dim, 64), nn.GELU(), nn.Dropout(0.20),
                                 nn.Linear(64, 64), nn.GELU(), nn.Dropout(0.10), nn.Linear(64, 1))

    def forward(self, xq, xc):
        return self.mlp(torch.cat([self.q(xq), xc], dim=1)).squeeze(-1)


def pcc_torch(a, b, eps=1e-8):
    a, b = a - a.mean(), b - b.mean()
    return torch.clamp((a * b).mean() / (torch.sqrt((a * a).mean() + eps) * torch.sqrt((b * b).mean() + eps) + eps), -1, 1)


def composite_loss_torch(pred, target):
    w_h, w_c, w_s = QNN_LOSS_WEIGHTS
    huber = nn.functional.huber_loss(pred, target, delta=1.0)
    corr = 1.0 - pcc_torch(pred, target) ** 2
    std = (torch.std(pred) - 1.0) ** 2
    return w_h * huber + w_c * corr + w_s * std


def pad_to_qubits(Z, n_qubits=QNN_QUBITS):
    extra = (-Z.shape[1]) % n_qubits
    return np.pad(Z, ((0, 0), (0, extra))) if extra else Z


def qnn_fit_predict(X_tr, y_tr, g_tr, X_te, epochs=None):
    epochs = epochs or QNN_EPOCHS_CV
    set_global_seed(SEED); torch.manual_seed(SEED)
    # 1) Scaling + PCA fitted on the training part only
    sx = StandardScaler().fit(X_tr)
    pca = PCA(n_components=QNN_TARGET_DIM, random_state=SEED).fit(sx.transform(X_tr))
    Z_tr = pca.transform(sx.transform(X_tr)).astype(np.float32)
    Z_te = pca.transform(sx.transform(X_te)).astype(np.float32)
    # 2) Classical branch: PCA components + grouped out-of-fold kNN meta-feature
    k_tr, k_te = knn_oof_feature(Z_tr, y_tr, g_tr, Z_te, QNN_KNN_NEIGHBORS)
    sk = StandardScaler().fit(k_tr)
    C_tr = np.hstack([Z_tr, sk.transform(k_tr)]).astype(np.float32)
    C_te = np.hstack([Z_te, sk.transform(k_te)]).astype(np.float32)
    Q_tr, Q_te = pad_to_qubits(Z_tr), pad_to_qubits(Z_te)
    # 3) Standardised target and validation split by protein cluster
    sy = StandardScaler().fit(y_tr.reshape(-1, 1))
    y_tr_s = sy.transform(y_tr.reshape(-1, 1)).ravel().astype(np.float32)
    tr, va = grouped_holdout(len(Q_tr), g_tr, QNN_VAL_SIZE)

    t = lambda a: torch.tensor(a, dtype=torch.float32)
    Qtr, Ctr, ytr = t(Q_tr[tr]), t(C_tr[tr]), t(y_tr_s[tr])
    Qva, Cva, yva = t(Q_tr[va]), t(C_tr[va]), t(y_tr_s[va])

    qnn, q_out = build_qnn(QNN_QUBITS, Q_tr.shape[1])
    net = HybridRegressor(qnn, q_out, C_tr.shape[1])
    opt = torch.optim.AdamW(net.parameters(), lr=QNN_LR, weight_decay=QNN_WD)
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=3, min_lr=1e-5)
    loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(Qtr, Ctr, ytr),
                                         batch_size=QNN_BATCH_SIZE, shuffle=True)

    best, bad, best_state = float("inf"), 0, None
    for ep in range(1, epochs + 1):
        t0 = time.time()
        net.train()
        for qb, cb, yb in loader:
            opt.zero_grad()
            loss = composite_loss_torch(net(qb, cb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            opt.step()
        net.eval()
        with torch.no_grad():                                   # validation set, never the test set
            pv = net(Qva, Cva)
            val = composite_loss_torch(pv, yva).item()
        sch.step(val)
        print(f"    época {ep:02d} | val={val:.4f} | PCC val≈{pcc_torch(pv, yva).item():.3f} | {time.time() - t0:.0f} s")
        if val + 1e-6 < best:
            best, bad = val, 0
            best_state = {k: v.detach().clone() for k, v in net.state_dict().items()}
        else:
            bad += 1
            if bad >= QNN_PATIENCE:
                print("    parada temprana")
                break
    if best_state is not None:
        net.load_state_dict(best_state)
    net.eval()
    with torch.no_grad():
        pred_s = net(t(Q_te), t(C_te)).numpy()
    return sy.inverse_transform(pred_s.reshape(-1, 1)).ravel()


qc_draw, *_ = build_circuit(QNN_QUBITS, QNN_TARGET_DIM)
print(f"Circuito: {QNN_QUBITS} qubits, {qc_draw.num_parameters - QNN_TARGET_DIM} parámetros entrenables, "
      f"profundidad {qc_draw.depth()}")
print(qc_draw.draw(output="text", fold=120))

print("\nQNN")
evaluate("QNN", qnn_fit_predict, features=X_qnn, test_kwargs={"epochs": QNN_EPOCHS_TEST})

---
# Parte VIII — Comparación y resultados

### 12 · Tabla, gráficos y registro

Resume los tres modelos (media ± desviación de la validación cruzada y prueba fija), dibuja la comparación y las predicciones de la prueba fija, y guarda todo en `results/`:

- `resultados.json`: métricas por partición, medias y prueba fija, junto con la configuración;
- `predicciones_prueba.csv`: predicciones de cada modelo para cada mutación de la prueba fija;
- `comparacion.png` y `dispersion_prueba.png`.

In [ ]:
rows = []
for name, r in RESULTS.items():
    rows.append({"Modelo": name,
                 "PCC (CV)": f"{r['cv']['PCC'][0]:.3f} ± {r['cv']['PCC'][1]:.3f}",
                 "RMSE (CV)": f"{r['cv']['RMSE'][0]:.3f} ± {r['cv']['RMSE'][1]:.3f}",
                 "MAE (CV)": f"{r['cv']['MAE'][0]:.3f} ± {r['cv']['MAE'][1]:.3f}",
                 "PCC (prueba)": round(r["test"]["PCC"], 3),
                 "RMSE (prueba)": round(r["test"]["RMSE"], 3),
                 "MAE (prueba)": round(r["test"]["MAE"], 3)})
summary = pd.DataFrame(rows).set_index("Modelo")
try:
    display(summary)                                         # Jupyter / Colab
except NameError:
    print(summary.to_string())

# ── Comparison chart ────────────────────────────────────────────────────────
names = list(RESULTS)
colors = {"XGBoost": "#219BC3", "FCNN": "#00566C", "QNN": "#7D3C98"}
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, metric in zip(axes, ["PCC", "RMSE"]):
    cv_mean = [RESULTS[n]["cv"][metric][0] for n in names]
    cv_std = [RESULTS[n]["cv"][metric][1] for n in names]
    test = [RESULTS[n]["test"][metric] for n in names]
    xs = np.arange(len(names))
    ax.bar(xs - 0.18, cv_mean, 0.34, yerr=cv_std, capsize=4, color=[colors.get(n, "gray") for n in names],
           label="validación cruzada (media ± desv.)")
    ax.bar(xs + 0.18, test, 0.34, color=[colors.get(n, "gray") for n in names], alpha=0.45, label="prueba fija")
    ax.set_xticks(xs); ax.set_xticklabels(names); ax.set_title(metric + (" (kcal/mol)" if metric == "RMSE" else ""))
    ax.grid(axis="y", alpha=0.3)
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, "comparacion.png"), dpi=200); plt.show()

# ── Predicted vs experimental on the fixed split ────────────────────────────
y_test = y[TEST_IDX]
fig, axes = plt.subplots(1, len(names), figsize=(5 * len(names), 4.6), sharex=True, sharey=True)
lo, hi = float(y.min()) - 0.5, float(y.max()) + 0.5
for ax, n in zip(np.atleast_1d(axes), names):
    p = np.array(RESULTS[n]["pred_test"])
    ax.scatter(y_test, p, s=14, alpha=0.55, color=colors.get(n, "gray"))
    ax.plot([lo, hi], [lo, hi], "--", color="gray", lw=1)
    ax.set_title(f"{n} — PCC {RESULTS[n]['test']['PCC']:.3f} · RMSE {RESULTS[n]['test']['RMSE']:.2f}")
    ax.set_xlabel("ΔΔG experimental (kcal/mol)"); ax.grid(alpha=0.3)
np.atleast_1d(axes)[0].set_ylabel("ΔΔG predicho (kcal/mol)")
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, "dispersion_prueba.png"), dpi=200); plt.show()

# ── Save results ────────────────────────────────────────────────────────────
pd.DataFrame({"PDB": df["PDB"].values[TEST_IDX], "CHAIN": df["CHAIN"].values[TEST_IDX],
              "MUT": df["MUT"].values[TEST_IDX], "CLID": groups[TEST_IDX], "DDG": y_test,
              **{f"pred_{n}": RESULTS[n]["pred_test"] for n in names}}
             ).to_csv(os.path.join(RESULTS_DIR, "predicciones_prueba.csv"), index=False)

out = {"run_mode": RUN_MODE, "seed": SEED,
       "datos": {"mutaciones": int(len(df)), "clusters": int(len(np.unique(groups))),
                 "prueba_mutaciones": int(len(TEST_IDX)), "prueba_clusters": int(len(set(groups[TEST_IDX]))),
                 "dim_caracteristicas": int(X.shape[1]), "dim_caracteristicas_qnn": int(X_qnn.shape[1]),
                 "qnn_usa_prott5": QNN_USE_PROTT5},
       "protocolo": {"cv": f"GroupKFold({N_FOLDS}) por {GROUP_COL}", "prueba": f"GroupShuffleSplit({TEST_SIZE})"},
       "modelos": {n: {k: v for k, v in r.items() if not k.startswith("pred_")} for n, r in RESULTS.items()}}
with open(os.path.join(RESULTS_DIR, "resultados.json"), "w", encoding="utf-8") as f:
    json.dump(out, f, ensure_ascii=False, indent=2)
print(f"Resultados guardados en {os.path.abspath(RESULTS_DIR)}")